In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


**CWRU dataset:** Smith, W. A., & Randall, R. B. (2015). Rolling element bearing diagnostics using the Case Western Reserve University data: A benchmark study. *Mechanical Systems and Signal Processing*, 64–65, 100–131.


# Feature Practice — CWRU 베어링 내륜 결함 진단

FFT · Filtering · Time-Frequency · Envelope 도구를 **하나의 실제 데이터 (CWRU 베어링 가속도 신호)** 위에서 파이프라인으로 연결한다.

**목표**
- 정상 신호 (`v_n`) 와 내륜 결함 (inner race, IR) 신호 (`v_f`) 를 비교한다.
- 시간 영역 → 주파수 영역 → 공진 대역 격리 → 포락선 복조 (demodulation) 순서로 단계를 따라간다.
- 각 단계에서 정상/결함 차이가 어떻게 부각되는지 관찰하고, 마지막에 **특징 벡터 (feature vector)** 로 정량화한다.

**데이터 출처**: Case Western Reserve University Bearing Data Center (내륜 결함 1772 rpm · 정상 약 1796 rpm, 12 kHz 샘플링, Drive-End 가속도).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import utils


## 실습 1. 베어링 특성 주파수 계산

베어링 결함이 발생하면, **회전 1바퀴당 결함부가 하중 구간을 지나가는 횟수**가 이론적으로 결정된다. CWRU Drive-End 베어링(SKF 6205)의 기하학적 상수는 다음과 같다.

| 결함 위치 | 특성 주파수 (shaft frequency 기준 배수) |
| --- | --- |
| 내륜 (Inner Race, BPFI) | 5.4152 |
| 외륜 (Outer Race, BPFO) | 3.5848 |

축 회전 속도 $f_{shaft} = \text{rpm}/60$을 곱하면 절대 주파수(Hz)가 나온다. 이 값이 **포락선 스펙트럼에서 기대되는 피크 위치**가 된다.


In [ ]:
rpm = 1772
F_shaft = rpm / 60
print('Shaft frequency:', F_shaft, 'Hz')

# Drive-End 베어링 특성 주파수
DE_BPFI = 5.4152 * F_shaft
DE_BPFO = 3.5848 * F_shaft
print('DE Inner Race Fault Frequency (BPFI):', DE_BPFI, 'Hz')
print('DE Outer Race Fault Frequency (BPFO):', DE_BPFO, 'Hz')

# Fan-End 베어링 (참고)
FE_BPFI = 4.9469 * F_shaft
FE_BPFO = 3.0530 * F_shaft
print('FE Inner Race Fault Frequency:', FE_BPFI, 'Hz')
print('FE Outer Race Fault Frequency:', FE_BPFO, 'Hz')


**관찰**
- Shaft 주파수는 약 **29.5 Hz**, DE 내륜 결함 주파수 **BPFI ≈ 160 Hz**이다.
- 이 노트북에서는 내륜 결함 데이터만 다루므로, 이후 `DE_BPFI` 및 그 배음(2배, 3배, …)이 포락선 스펙트럼의 **진단 지표**가 된다.
- 단, **슬립(slip)** 때문에 실제 피크는 이론값에서 보통 1~2 % 벗어나고 운전 조건에 따라 더 클 수 있다.


---


## 실습 2. 신호 로드와 파형 비교

CSV 파일 두 개를 읽어 1-D 배열로 만든다. 컬럼 0은 샘플 인덱스, 컬럼 1이 가속도(g) 값이다.

- `data/data_normal.csv` → 정상 베어링 (`v_n`)
- `data/data_fault_DE_IR.csv` → 내륜 결함 (`v_f`)

샘플링 주파수는 **12 kHz** (CWRU 규격). 두 신호는 길이가 동일(10초)하므로 시간축을 공유할 수 있다.


In [ ]:
fs = 12000

data = np.array(pd.read_csv('./data/data_normal.csv'))
v_n = data[:, 1]

data = np.array(pd.read_csv('./data/data_fault_DE_IR.csv'))
v_f = data[:, 1]

# 시간 벡터: arange(N)/fs 로 정확히 N 개 샘플에 매핑 (오프셋 없이)
t_n = np.arange(len(v_n)) / fs
t_f = np.arange(len(v_f)) / fs

# 파형 비교 (Fault peak 기준 자동 스케일)
y_lim = np.max(np.abs(v_f)) * 1.1

plt.figure(figsize=(10, 4))
plt.plot(t_n, v_n, color='C0', label='Normal', alpha=0.9)
plt.plot(t_f, v_f, color='C1', label='Fault (DE IR)', alpha=0.7)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (g)')
plt.ylim([-y_lim, y_lim])
plt.legend()
plt.tight_layout()
plt.show()

print(f'RMS  Normal = {np.sqrt(np.mean(v_n**2)):.4f} g')
print(f'RMS  Fault  = {np.sqrt(np.mean(v_f**2)):.4f} g')
print(f'Peak Normal = {np.max(np.abs(v_n)):.4f} g')
print(f'Peak Fault  = {np.max(np.abs(v_f)):.4f} g')


**관찰**
- 결함 신호는 정상 대비 **RMS 약 1.2배**, **Peak 약 2배** 수준으로 커진다.
- RMS보다 Peak가 더 크게 벌어진다는 점이 중요하다 — 결함이 **임펄스성(peak-y)** 이라는 뜻이다.
- 하지만 시간 파형만 봐서는 "진폭이 조금 커졌다" 외에 정보가 부족하다. 주파수 영역으로 가자.


---


## 실습 3. 스펙트럼 비교

두 신호의 단측 진폭 스펙트럼을 구해 나란히 비교한다. 베어링 결함의 임펄스는 **구조물 공진 대역(수 kHz)** 을 주기적으로 가진해 그 대역 에너지를 키운다. 따라서 저주파가 아니라 **고주파 공진 대역**을 먼저 확인해야 한다.


In [ ]:
f_n, A_n = utils.fft(v_n - np.mean(v_n), fs)
f_f, A_f = utils.fft(v_f - np.mean(v_f), fs)

plt.figure(figsize=(10, 4))
plt.plot(f_n, A_n, color='C0', label='Normal', alpha=0.9)
plt.plot(f_f, A_f, color='C1', label='Fault (DE IR)', alpha=0.7)
plt.xlabel('Frequency (Hz)')
plt.ylabel('|Y|')
plt.xlim([0, fs / 2])
plt.legend()
plt.tight_layout()
plt.show()


**관찰**
- 결함 신호는 **약 3~6 kHz 사이**의 브로드밴드 대역에서 정상 대비 **열 배 이상**(3.5 ~ 5.5 kHz 대역 스펙트럼 RMS 로 약 17배) 올라온다 — 이 구간이 구조물 공진 대역이다.
- 저주파(<500 Hz)의 BPFI(≈ 160 Hz) 자리는 이 그림에서 다른 선과 구별되지 않는다(0–500 Hz 가 그림 폭의 1/12 이다). `plt.xlim([0, 500])` 으로 확대하면 BPFI 자리에 Fault 의 작은 선이 있지만, 같은 구간에 그보다 높은 Fault 선이 여섯 개 있고 Normal 도 바로 옆(약 161 Hz)에 그보다 조금 높은 선을 가져 BPFI 를 구별할 수 없다. 직접 FFT로는 **결함 진단 정보가 고주파 캐리어에 실려** 있어서, 저주파에서는 묻힌다.
- 해결책: **공진 대역만 대역통과 필터로 추출** → **힐베르트 포락선을 구한 뒤** → **다시 주파수 분석**하면 저주파의 BPFI 가 드러난다. 다음 실습에서 차례로 확인한다.


---


## 실습 4. 대역통과 필터링 — 공진 대역 격리

실습 3에서 확인한 공진 대역 **[3500, 5500] Hz**만 통과시킨다. Butterworth 4차 zero-phase (`sosfiltfilt`)이므로 위상 왜곡은 없다.

필터링된 신호는 **공진 주파수로 진동하는 캐리어(carrier)** 가 **결함 임펄스의 리듬(BPFI, ~160 Hz)** 으로 진폭 변조된 형태가 된다. 이 AM 구조에서 변조 정보를 추출하는 것이 다음 단계(포락선 복조)이다.


In [ ]:
f_low, f_high = 3500, 5500

v_filter_n = utils.filtering_zerophase(v_n, fs, 'band', f_low=f_low, f_high=f_high)
v_filter_f = utils.filtering_zerophase(v_f, fs, 'band', f_low=f_low, f_high=f_high)

# 자동 스케일 (fault 기준)
y_lim_filter = np.max(np.abs(v_filter_f)) * 1.1

plt.figure(figsize=(10, 4))
plt.plot(t_n, v_filter_n, color='C0', label='Normal', alpha=0.9)
plt.plot(t_f, v_filter_f, color='C1', label='Fault (DE IR)', alpha=0.7)
plt.xlabel('Time (s)')
plt.ylabel('Filtered amplitude (g)')
plt.ylim([-y_lim_filter, y_lim_filter])
plt.title(f'Bandpass [{f_low}, {f_high}] Hz')
plt.legend()
plt.tight_layout()
plt.show()


**관찰**
- 필터링 후 **Fault 파형의 임펄스 구조가 선명해진다** — 충격 덩어리가 위로 솟은 뾰족한 선으로 보인다. 결함 주기는 BPFI 주기 (약 6.25 ms) 이지만 충격의 세기가 축의 회전 각도에 따라 크게 달라져, 10 초를 한 화면에 담은 이 그림에서는 큰 충격만 뾰족한 선으로 두드러진다 (6 ms 간격은 이 그림 폭에서 1 픽셀보다 좁아 낱낱이 보이지 않는다).
- Normal은 같은 대역에서 거의 평평한 잡음 수준이다.
- 이제 이 Fault 신호의 **포락선(envelope)** 만 추출하면, 고주파 캐리어는 제거되고 **결함 충격의 느린 리듬**(BPFI 와 축 회전 변조)이 남는다.


---


## 실습 5. 포락선 스펙트럼 — BPFI 확인

**힐베르트 포락선**으로 필터링된 신호의 순시 진폭을 구한 뒤, DC 성분을 빼고 FFT 한다 (`utils.hilbert_envelope`).

포락선 스펙트럼의 저주파 구간 (0~500 Hz) 에서 다음을 기대한다.
- **BPFI ≈ 160 Hz** 근처에 뚜렷한 피크
- 그 **2배(≈320 Hz), 3배(≈480 Hz)** 배음(harmonic)
- 축 회전 주파수의 정수배 선과, BPFI 양옆에 축 회전 주파수(≈ 29.5 Hz) 간격으로 늘어선 사이드밴드 (내륜 결함 특유의 현상)


In [ ]:
# Hilbert envelope (utils 래퍼 사용)
v_filter_env_n = utils.hilbert_envelope(v_filter_n)
v_filter_env_f = utils.hilbert_envelope(v_filter_f)

# DC 성분 제거 후 FFT
f_env_n, A_env_n = utils.fft(v_filter_env_n - np.mean(v_filter_env_n), fs)
f_env_f, A_env_f = utils.fft(v_filter_env_f - np.mean(v_filter_env_f), fs)

plt.figure(figsize=(10, 4))
plt.plot(f_env_n, A_env_n, color='C0', label='Normal', alpha=0.9, zorder=3)
plt.plot(f_env_f, A_env_f, color='C1', label='Fault (DE IR)', alpha=0.7, zorder=2)

# shaft harmonic (검은색), BPFI harmonic (빨강) — 첫 번째에만 label 달아 legend 중복 방지
# 가는 점선을 데이터 뒤(zorder=1)에 두어 피크 높이를 가리지 않게 한다
for n in range(1, 4):
    plt.axvline(n * F_shaft, color='k', linestyle=':', lw=0.8, alpha=0.5, zorder=1,
                label='Shaft harmonic' if n == 1 else None)
    plt.axvline(n * DE_BPFI, color='C3', linestyle=':', lw=0.8, alpha=0.7, zorder=1,
                label='BPFI harmonic' if n == 1 else None)

plt.xlabel('Frequency (Hz)')
plt.ylabel('|Y|')
plt.xlim([0, 600])
plt.legend()
plt.tight_layout()
plt.show()


**관찰**
- Fault 포락선 스펙트럼에서 **BPFI ≈ 160 Hz** 의 피크가 그림 전체에서 가장 높다 (약 0.0175). 배음 2·BPFI ≈ 320 Hz (약 0.003) 는 바로 옆 선들보다 조금 높을 뿐이다. 3·BPFI ≈ 480 Hz (약 0.0015) 는 바닥보다 높지만 주변 잡음 피크와 높이가 비슷하다(그 왼쪽 약 453 Hz 의 높은 선은 BPFI 의 정수배가 아니다).
- 160 Hz 아래의 피크들은 축 회전 주파수(약 29.5 Hz)의 정수배(30 · 59 · 89 · 118 Hz)와, BPFI 양옆에 축 회전 주파수 간격으로 늘어선 사이드밴드 가운데 아래쪽 것(17 · 42 · 71 · 101 · 130 Hz)이다 — 내륜 결함의 충격 세기가 축 회전 주기로 오르내려 생긴 성분이라 결함을 뒷받침한다. 다만 결함 판정은 이론이 지정한 BPFI 정수배 위치에서 한다.
- Normal은 같은 주파수대에서 BPFI 피크가 없다.
- **대역통과 필터로 공진 대역을 격리하면 진단 성분이 더 깨끗하게 드러난다.** 원 신호에서 바로 포락선을 구해도 이 신호에서는 BPFI 피크가 가장 높지만, 축 회전 성분(29.6 Hz)이 BPFI 의 약 85 % 높이로 함께 서고 3·BPFI 는 바닥의 약 2.4 배로 낮아진다.
- 이것이 베어링 진단의 핵심 파이프라인이다 — **대역통과 필터 → 힐베르트 포락선 → 주파수 분석**.


---


## 실습 6. 특징 벡터 추출과 Fault/Normal Ratio

지금까지는 그림으로 비교했다. 실제 진단 시스템은 **정량화된 숫자(특징, feature)** 가 필요하다. `utils.feature`는 앞선 파이프라인을 한 함수로 묶는다. 다만 대역통과 필터는 인과 버전 `utils.filtering` 을 쓰고 앞 100 샘플을 0 으로 둔다 — 크기만 읽는 특징이라 위상 지연은 결과를 거의 바꾸지 않는다.

| Feature 그룹 | 내용 |
| --- | --- |
| `RMS / Skew / Kurt / CF` | 원 신호의 시간 영역 통계 |
| `Band1` | 원 신호에서 공진 대역 `[3500, 5500] Hz` RMS |
| `RMS_filter / …` | 대역통과 필터링 후 신호의 시간 영역 통계 |
| `Band1_env, Band2_env, Band3_env` | 포락선 스펙트럼에서 BPFI, 2·BPFI, 3·BPFI 주변 에너지 |

**`band_from_envelope` 설계**

베어링 결함 주파수는 슬립 때문에 이론값에서 **보통 1~2 %** 벗어나고 운전 조건에 따라 더 크게 흔들릴 수 있다. 회전 속도도 조금씩 변한다. 단일 주파수 bin 한 개만 읽으면 피크가 옆으로 새어 놓칠 수 있다. 그래서 **1배/2배/3배 harmonic 각각에 ±10% 윈도우**를 잡고 그 안 에너지를 적분한다.


In [ ]:
band_filter = [3500, 5500]  # 공진 분석 대역 = 실습 4 와 동일

# BPFI 및 배음 각각 ±10% 윈도우
band_from_envelope = np.array([
    [1 * DE_BPFI * 0.9, 1 * DE_BPFI * 1.1],
    [2 * DE_BPFI * 0.9, 2 * DE_BPFI * 1.1],
    [3 * DE_BPFI * 0.9, 3 * DE_BPFI * 1.1],
])

print('Bandpass filter range:', band_filter)
print('Envelope harmonic bands (Hz):')
print(band_from_envelope)


In [ ]:
feature_n, feature_name = utils.feature(v_n, fs, band_filter, band_from_envelope)
feature_f, feature_name = utils.feature(v_f, fs, band_filter, band_from_envelope)

# 값의 크기가 다섯 자릿수 넘게 걸치므로 세로축을 로그로 (왜도는 음수가 될 수 있어 절댓값)
x = np.arange(len(feature_name))
plt.figure(figsize=(10, 5))
plt.plot(x, np.abs(feature_n), '-o', color='C0', label='Normal')
plt.plot(x, np.abs(feature_f), '-x', color='C1', label='Fault (DE IR)')
plt.yscale('log')
plt.xticks(x, feature_name, rotation=45, ha='right')
plt.ylabel('|Feature value|')
plt.legend()
plt.tight_layout()
plt.show()

**관찰 — 절대값 비교**
- 특징값이 약 0.00003 (포락선 대역 특징) 에서 7 (파고율) 까지 다섯 자릿수 넘게 걸쳐 있어 세로축을 로그로 그렸다. 로그 축에서는 아래위 한 칸이 10 배다.
- 시간 통계 네 자리 (`RMS`, `Skew`, `Kurt`, `CF`) 에서는 `Skew` 를 빼면 정상과 결함이 거의 붙어 있고 (`Skew` 는 두 값이 모두 0 근처라 간격에 뜻이 없다), 공진 대역·포락선 대역 특징 (`Band1`, `RMS_filter`, `Band*_env`) 으로 갈수록 두 계열이 벌어진다.
- 특징마다 단위와 크기가 달라 절댓값끼리 비교하기는 어렵다. 다음 셀에서 특징마다 «결함 / 정상» **비율** 하나로 압축해 본다.

In [ ]:
feature_ratio = np.abs(feature_f) / np.abs(feature_n)

plt.figure(figsize=(10, 5))
bars = plt.bar(x, feature_ratio, color='C0', width=0.6)
plt.bar_label(bars, labels=[f'{r:.1f}' if r >= 1 else f'{r:.2f}' for r in feature_ratio])
plt.axhline(y=1, color='C3', linestyle='--', label='ratio = 1')
plt.yscale('log')
plt.xticks(x, feature_name, rotation=45, ha='right')
plt.ylabel('|Fault| / |Normal|')
plt.legend()
plt.tight_layout()
plt.show()

# 숫자로도 확인
for name, r in zip(feature_name, feature_ratio):
    print(f'{name:<14s}: {r:7.2f}')

**관찰 — 비율 비교 (핵심)**
- 시간 통계는 `RMS` 1.2 · `Kurt` 1.4 · `CF` 1.7 로 결함에서 조금 커지는 데 그친다. 임펄스성 지표인 `Kurt` · `CF` 가 `RMS` 보다 조금 더 반응한다.
- 공진 대역 특징 `Band1` (17.2) · `RMS_filter` (17.1) 과 포락선 대역 특징 `Band1_env` (35.6) · `Band2_env` (21.2) · `Band3_env` (11.7) 은 10 배 이상 반응한다. 그중 BPFI 1 차 대역의 `Band1_env` 가 가장 크다.
- `Skew` (0.02) 와 `Skew_filter` (23.2) 는 정상·결함 값이 모두 0 근처 (`Skew_filter` 는 0.0003 과 −0.008) 라 비율 자체에 뜻이 없다 — 0 근처 값끼리의 비율은 작은 차이도 크게 부풀린다.
- 결론: **공진 대역 필터링과 포락선 복조가 베어링 진단의 핵심 단계**이다. 시간 통계 단독은 분별력이 낮다.

---
